# 04 — Sampling : temperature, top-k, top-p

**Ton premier notebook.** Mode d'emploi : clique dans une cellule et fais
`Shift+Entrée` pour l'exécuter — le résultat s'affiche juste dessous, et les
variables restent en vie d'une cellule à l'autre. En haut à droite, VS Code te
demandera de choisir un *kernel* : prends le Python du `.venv` de ce dossier.

**Rappel du concept** (le brief complet est dans la conversation / PROGRESSION.md) :
à chaque étape, le modèle ne produit pas un mot, mais une **probabilité pour
chaque token de son vocabulaire**. Le *sampling*, c'est la règle du tirage au
sort dans cette liste. Les paramètres ci-dessous règlent ce tirage — ils vivent
**hors du modèle**, dans le serveur d'inférence. Même modèle, réglages différents
→ comportements très différents. C'est ce qu'on va constater.

In [2]:
import httpx

OLLAMA_URL = "http://192.168.1.57:11434"
MODEL = "qwen3:4b-instruct-2507-q4_K_M"


def demander(prompt, **options):
    """Pose une question (sans historique) avec des options de sampling.

    **options est une nouveaute Python : tous les arguments nommes passes
    a l'appel (temperature=0.8, top_k=40...) sont ramasses dans un
    dictionnaire `options`, qu'on transmet tel quel a Ollama.
    """
    # GARDE-FOU (ajoute apres incident) : un sampling debride peut partir
    # en charabia sans jamais produire de token d'arret -> generation
    # quasi infinie, GPU a fond, timeout client (vecu le 19/07 :
    # temperature=1.5 sur une question ouverte). num_predict borne le
    # nombre de tokens generes — toute app LLM serieuse fixe cette limite
    # (max_tokens chez OpenAI/Anthropic). Surchargeable : num_predict=...
    options = {"num_predict": 300, **options}
    reponse = httpx.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "options": options,
        },
        timeout=120,
    )
    return reponse.json()["message"]["content"]


# Test rapide : si cette cellule affiche une salutation, tout marche.
demander("Dis bonjour en un mot. Reponds par ce mot seul, rien d'autre.", temperature=0.0)

'Salut'

## Expérience 1 — la temperature

- `temperature = 0` : toujours prendre le token **le plus probable**. Sortie
  quasi identique à chaque essai.
- `temperature ≈ 0.7-0.8` : le réglage « naturel » par défaut — les favoris
  gagnent souvent, les outsiders parfois.
- `temperature ≥ 1.5` : la loterie s'aplatit, les tokens improbables sortent —
  créativité, puis n'importe quoi.

**Piège découvert en préparant ce notebook** : le modèle embarque ses propres
défauts de sampling dans son Modelfile (ici `temperature 0.7, top_k 20,
top_p 0.8` — visibles via l'API `/api/show`). Une option absente de la requête
retombe sur le défaut du Modelfile. Régler *seulement* la temperature laisse
donc `top_p=0.8` actif, qui coupe la loterie aux favoris et masque l'effet
d'une haute temperature ! D'où le `top_p=1.0, top_k=0` (= désactivés) dans la
cellule suivante, pour isoler la variable qu'on étudie.

La cellule suivante pose **la même question 3 fois à chaque réglage**.
Compare la variété des réponses entre les trois blocs.

In [4]:
QUESTION = "qu'est ce que l'hydrogen en une phrase"

for temp in [0.0, 0.8, 1.5]:
    print(f"--- temperature = {temp} ---")
    for _ in range(3):  # _ : convention pour "je n'utilise pas cette variable"
        # top_p=1.0 et top_k=0 neutralisent les defauts du Modelfile
        # pour observer l'effet de la temperature seule.
        print("  ", demander(QUESTION, temperature=temp, top_p=1.0, top_k=0))

--- temperature = 0.0 ---
   L'hydrogène est l'élément chimique le plus léger et le plus abondant dans l'univers, composé d'un seul proton et d'un seul électron.
   L'hydrogène est l'élément chimique le plus léger et le plus abondant dans l'univers, composé d'un seul proton et d'un seul électron.
   L'hydrogène est l'élément chimique le plus léger et le plus abondant dans l'univers, composé d'un seul proton et d'un seul électron.
--- temperature = 0.8 ---
   L'hydrogène est l'élément chimique le plus léger et le plus abondant dans l'univers, composé d'un seul proton et d'un seul électron.
   L'hydrogène est l'élément chimique le plus simple et le plus abondant dans l'univers, composé d'un seul proton et d'un seul électron.
   L'hydrogène est l'élément chimique le plus léger et le plus abondant dans l'univers, formé par la fusion de protons dans les étoiles.
--- temperature = 1.5 ---
   L'hydrogène est un element chimique avec le numéro atomic 1, qui est le plus léger et le plus abondan

## Expérience 2 — top-k et top-p : couper la queue de la loterie

La temperature règle *les poids* du tirage ; top-k et top-p règlent *qui a le
droit de participer* :

- **top-k** : ne garder que les `k` tokens les plus probables (top_k=1 = le
  favori gagne toujours, quelle que soit la temperature !).
- **top-p** (*nucleus sampling*) : garder juste assez de tokens pour couvrir
  `p` de probabilité cumulée (top_p=0.5 → seulement les favoris qui pèsent
  ensemble 50 %). Plus adaptatif que top-k : la taille de la liste varie selon
  que le modèle est sûr de lui ou hésite.

### >>> EXERCICE (à toi d'écrire, dans la cellule suivante)

En t'inspirant de la boucle de l'expérience 1 :

1. Compare `temperature=1.5` seul **vs** `temperature=1.5, top_k=1` —
   3 essais chacun. Que conclus-tu sur qui a le dernier mot ?
2. À `temperature=1.2`, compare `top_p=0.5` vs `top_p=1.0`.
3. Bonus : ajoute `seed=42` avec `temperature=0.8` et lance deux fois —
   surprise ? (le *seed* fixe le générateur aléatoire du tirage)

In [12]:
# >>> Ton code ici — demander() accepte temperature=, top_k=, top_p=, seed=
QUESTION = "qu'est ce que l'hydrogen en une phrase"

compare = {
    0 : {"temperature": 1.5, "top_p": 1, "top_k": 0},
    1 : {"temperature": 1.5, "top_p": 1, "top_k": 1},
    2 : {"temperature": 1.2, "top_p": 0.5, "top_k": 0},
    3 : {"temperature": 1.2, "top_p": 1.0, "top_k": 0},
    4 : {"temperature": 0.8, "top_p": 1, "top_k": 0, "seed": 42},
}
for i in range(5):
    option = compare[i]
    print(f"--- {option} ---")
    for _ in range(3):
        print("  ", demander(QUESTION, **option))

--- {'temperature': 1.5, 'top_p': 1, 'top_k': 0} ---
   L'hydrogène est un élément chimique, présent dans l'univers, et il est le plus léger et le plus abondant des corps purs.
   L'hydrogène est l'élément chimique le plus léger et le plus ubiquitaire de l'Univers, composé d'un seul proton et d'un seul électron, et essentiel à la formation des molécules organiques et des réactions de combustion.
   L'hydrogène est un élément chimique light (symbole H) et le plus simple de tous, composé d'un seul proton et d'un électron, présent dans l'univers et essentiel à la formation des molécules comme l'eau.
--- {'temperature': 1.5, 'top_p': 1, 'top_k': 1} ---
   L'hydrogène est l'élément chimique le plus léger et le plus abondant dans l'univers, composé d'un seul proton et d'un seul électron.
   L'hydrogène est l'élément chimique le plus léger et le plus abondant dans l'univers, composé d'un seul proton et d'un seul électron.
   L'hydrogène est l'élément chimique le plus léger et le plus abondant

## À retenir

- Le modèle produit des **probabilités**, pas du texte : tout l'« aléatoire »
  vient du tirage, réglable de l'extérieur.
- `temperature=0` ≠ garanti identique à 100 % (parallélisme GPU, batching…),
  mais quasi.
- Réglage pratique : **basse temperature** pour l'extraction/le code/le
  factuel, **moyenne** pour le chat, **haute** pour le brainstorming — jamais
  haute pour un agent qui doit produire du JSON fiable.

Quand tu as fini : note tes observations ici même (double-clic sur une cellule
markdown pour l'éditer), et on cochera la case dans PROGRESSION.md.